# Colab Setup — TFM Pose 6-DoF

**Este notebook configura TODO el entorno en Google Colab.**

Ejecutar: **Runtime > Run all** (con GPU T4 activada)

### Flujo de trabajo
1. Desarrollar en local (Cursor + git + push)
2. Abrir este notebook en Colab (carga celdas frescas desde GitHub)
3. Ejecutar evaluaciones con GPU
4. Resultados se guardan en Google Drive

## 0. Configuracion Global

In [ ]:
import os, sys, time
from pathlib import Path

REPO_URL   = "https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git"
REPO_DIR   = "/content/repo_tfm"
LOCAL_DATA = "/content/datasets"
USE_GDRIVE = True

print(f"REPO_DIR   = {REPO_DIR}")
print(f"LOCAL_DATA = {LOCAL_DATA}")

## 1. GPU + Entorno

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU requerida! Ve a Runtime > Change runtime type > T4 GPU")

!df -h /content | tail -1

## 2. Clonar repositorio

In [ ]:
if os.path.exists(REPO_DIR):
    print("Repo existe, actualizando...")
    !cd {REPO_DIR} && git pull
else:
    print("Clonando...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -3

## 3. Instalar dependencias (uv)

Usamos [uv](https://github.com/astral-sh/uv) de Astral: 10-100x mas rapido que pip, mejor resolucion de dependencias.

In [ ]:
!pip install -q uv 2>&1 | tail -1

# Solo lo que nuestro codigo importa directamente
!uv pip install --system -q trimesh diffusers accelerate transformers scikit-learn

# Verificar numpy (algunas deps transitivas pueden bajarlo)
import numpy as np
if int(np.__version__.split('.')[0]) < 2:
    print(f"numpy {np.__version__} < 2.0, restaurando...")
    !uv pip install --system -q "numpy>=2.0"
    import importlib; importlib.reload(np)

print(f"numpy {np.__version__} OK")

# Verificar imports del proyecto
from src.utils.lie_groups import so3_exp, se3_exp
from src.utils.rotations import matrix_to_6d, sixd_to_matrix
from src.utils.metrics import compute_add, compute_adds
from src.planning.diffusion_policy import DiffusionGraspPlanner
from src.pipeline import BinPickingPipeline, PipelineConfig
print("Todos los imports OK")

## 4. Google Drive + Datasets BOP

- Zips en **Drive** (persistentes entre sesiones)
- Extraccion en **disco local** `/content/` (rapido, ~80 GB libres)
- Si los zips ya estan en Drive, solo extrae (~5 min)

In [ ]:
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ZIPS = "/content/drive/MyDrive/TFM/datasets_zips" if USE_GDRIVE else "/content/zips"
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights" if USE_GDRIVE else "/content/weights"

for d in [DRIVE_ZIPS, LOCAL_DATA, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Symlink: repo/data/datasets -> /content/datasets
REPO_DATA = f"{REPO_DIR}/data/datasets"
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA):
    import shutil; shutil.rmtree(REPO_DATA)
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
os.symlink(LOCAL_DATA, REPO_DATA)
print(f"Symlink: {REPO_DATA} -> {LOCAL_DATA}")

In [ ]:
HF_BASE = "https://huggingface.co/datasets/bop-benchmark"

def download_and_extract(dataset, filename, url):
    zip_path = f"{DRIVE_ZIPS}/{filename}"
    extract_dir = f"{LOCAL_DATA}/{dataset}"
    os.makedirs(extract_dir, exist_ok=True)

    if os.path.exists(zip_path) and os.path.getsize(zip_path) > 1000:
        print(f"  [CACHE] {filename} ({os.path.getsize(zip_path)/1e6:.0f} MB)")
    else:
        print(f"  [DOWN] {filename}...")
        !wget -q --show-progress -c -O "{zip_path}" "{url}"

    print(f"  [UNZIP] Extrayendo...")
    t0 = time.time()
    !unzip -qo "{zip_path}" -d "{extract_dir}"
    print(f"  [OK] {time.time()-t0:.0f}s")

def reorganize_bop(dataset_dir, nested_name, camera_src="camera.json"):
    """Aplana la estructura BOP: dataset/nested/ -> dataset/"""
    nested = Path(dataset_dir) / nested_name
    if nested.exists():
        !cp -rn {nested}/* {dataset_dir}/ 2>/dev/null; true
    cam_src = Path(dataset_dir) / camera_src
    if cam_src.exists() and not (Path(dataset_dir) / "camera.json").exists():
        !cp {cam_src} {dataset_dir}/camera.json

def verify_dataset(dataset_dir, split):
    """Verifica que el dataset tenga imagenes RGB."""
    test_dir = Path(dataset_dir) / split
    if not test_dir.exists():
        print(f"  ERROR: {split}/ no encontrado")
        return False
    scenes = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
    if not scenes:
        print(f"  ERROR: sin escenas en {split}/")
        return False
    rgb_dir = test_dir / scenes[0] / "rgb"
    imgs = sorted(rgb_dir.glob("*.*")) if rgb_dir.exists() else []
    print(f"  {len(scenes)} escenas, primera: {scenes[0]}, {len(imgs)} imgs ({imgs[0].name if imgs else 'N/A'})")
    return len(imgs) > 0

print("Funciones listas.")

### 4a. YCB-Video (21 objetos domesticos, ~15 GB)

In [ ]:
print("=== YCB-Video ===")
download_and_extract("ycbv", "ycbv_base.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_base.zip")
download_and_extract("ycbv", "ycbv_models.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_models.zip")
download_and_extract("ycbv", "ycbv_test_all.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_test_all.zip")

YCBV = f"{LOCAL_DATA}/ycbv"
reorganize_bop(YCBV, "ycbv", "camera_uw.json")
verify_dataset(YCBV, "test")

### 4b. T-LESS (30 objetos industriales, ~8 GB)

In [ ]:
print("=== T-LESS ===")
download_and_extract("tless", "tless_base.zip", f"{HF_BASE}/tless/resolve/main/tless_base.zip")
download_and_extract("tless", "tless_models.zip", f"{HF_BASE}/tless/resolve/main/tless_models.zip")
download_and_extract("tless", "tless_test_primesense_all.zip", f"{HF_BASE}/tless/resolve/main/tless_test_primesense_all.zip")

TLESS = f"{LOCAL_DATA}/tless"
reorganize_bop(TLESS, "tless", "camera_primesense.json")
# T-LESS usa models_cad como directorio de modelos
!ln -sf models_cad {TLESS}/models 2>/dev/null; true
verify_dataset(TLESS, "test_primesense")

### 4c. Resumen de espacio

In [ ]:
print("=== Disco ===")
!df -h /content | tail -1
print()
!du -sh {LOCAL_DATA}/*/ 2>/dev/null
print()
!du -sh {DRIVE_ZIPS}/ 2>/dev/null

## 5. Verificar Dataset Loader

In [ ]:
from src.utils.dataset_loader import BOPDataset
import matplotlib.pyplot as plt

for name, split in [("ycbv", "test"), ("tless", "test_primesense")]:
    try:
        ds = BOPDataset(f"{REPO_DIR}/data/datasets/{name}", split=split)
        scenes = ds.get_scene_ids()
        img_ids = ds.get_image_ids(scenes[0])
        print(f"{name.upper()}: {len(scenes)} escenas, primera escena {scenes[0]} tiene {len(img_ids)} imgs (IDs {img_ids[0]}..{img_ids[-1]})")

        sample = ds.load_sample(scenes[0], img_ids[0])
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.imshow(sample['rgb']); ax1.set_title(f'{name.upper()} {scenes[0]} - RGB')
        ax2.imshow(sample['depth'], cmap='viridis'); ax2.set_title('Depth')
        plt.tight_layout(); plt.show()
        print(f"  RGB: {sample['rgb'].shape}, Depth: {sample['depth'].shape}")
    except Exception as e:
        print(f"{name.upper()}: ERROR — {e}")
    print()

## 6. Pesos Pre-entrenados

In [ ]:
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Symlink weights
repo_weights = f"{REPO_DIR}/weights"
if os.path.islink(repo_weights):
    os.unlink(repo_weights)
elif os.path.isdir(repo_weights):
    import shutil; shutil.rmtree(repo_weights)
os.symlink(WEIGHTS_DIR, repo_weights)

for model, url in [("foundationpose", "https://github.com/NVlabs/FoundationPose#pre-trained-weights"),
                    ("gdrnet", "https://github.com/shanice-l/gdrnpp_bop2022#model-zoo")]:
    d = f"{WEIGHTS_DIR}/{model}"
    os.makedirs(d, exist_ok=True)
    files = list(Path(d).glob("*"))
    status = f"{len(files)} archivos" if files else "vacio — descargar manualmente"
    print(f"{model}: {status}")
    print(f"  {url}")

## 7. Sanity Check — GPU + Pipeline

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
from src.utils.lie_groups import so3_exp, pose_from_Rt
from src.planning.diffusion_policy import DiffusionGraspPlanner, ConditionalUNet1D

# UNet1D forward pass en GPU
model = ConditionalUNet1D(action_dim=7, horizon=16, cond_dim=64).to(device)
x = torch.randn(4, 16, 7, device=device)
t = torch.randint(0, 100, (4,), device=device)
c = torch.randn(4, 64, device=device)
with torch.no_grad():
    out = model(x, t, c)
print(f"UNet1D: {out.shape} on {device}")

# Grasp planner
planner = DiffusionGraspPlanner(device=device)
R = so3_exp(np.array([0.1, -0.2, 0.3]))
T = pose_from_Rt(R, np.array([0.15, -0.08, 0.45]))
traj = planner.plan_grasp_heuristic(T)
print(f"Grasp trajectory: {traj.shape}")

# GPU benchmark
if device == "cuda":
    batch = torch.randn(64, 16, 7, device=device)
    ts = torch.randint(0, 100, (64,), device=device)
    conds = torch.randn(64, 64, device=device)
    for _ in range(10): model(batch, ts, conds)  # warmup
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(100): model(batch, ts, conds)
    torch.cuda.synchronize()
    print(f"Throughput: {6400/(time.time()-t0):.0f} samples/s")

print(f"\nSetup completo en {device.upper()}!")

## 8. Guardar resultados (utility)

In [ ]:
import json
from datetime import datetime

def save_experiment(name, results, figures=None):
    exp_dir = f"/content/drive/MyDrive/TFM/experiments/{name}" if USE_GDRIVE else f"{REPO_DIR}/experiments/{name}"
    os.makedirs(exp_dir, exist_ok=True)
    results['timestamp'] = datetime.now().strftime('%Y%m%d_%H%M%S')
    results['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    out_file = f"{exp_dir}/results_{results['timestamp']}.json"
    with open(out_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Guardado: {out_file}")
    if figures:
        for fig, fname in figures:
            fig.savefig(f"{exp_dir}/{fname}", dpi=150, bbox_inches='tight')
            print(f"Figura: {exp_dir}/{fname}")
    return exp_dir

save_experiment('test_setup', {'status': 'OK', 'pytorch': torch.__version__})
print("\nListo! Continua con 01_foundationpose_eval.ipynb")